# PySpark 7: monitoring, performance and tuning

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-07-monitoring-and-performance.ipynb)

A slow job is usually a plan problem: a skewed key, a shuffle that could have been a broadcast, or too many small files. This module shows how to read the plan, the Spark UI and the event log, how to measure a change, and then the levers worth pulling: shuffle partitions and AQE, skew and salting, join strategy, file layout, caching, and executor sizing. Eight "Try it" problems and a checklist close it out.

*Last reviewed: November 2026, against Apache Spark 4.3 (runnable cells pinned to PySpark 4.0).*

**You will learn to:**

- Read a physical plan and find the expensive node
- Spot skew, spill and small-file problems in the Spark UI and the event log
- Measure a tuning change before and after, on the same data
- Size shuffle partitions and use adaptive query execution
- Fix skew with AQE skew join or salting, and choose a join strategy
- Lay out files, cache deliberately, and size executors and memory

**Before you start:** PySpark 1 to 6.

## Setup and sample data

This module runs on the same four tables as PySpark 1, so each module can run on its own, plus a larger, deliberately skewed orders table for the tuning lessons. The setup cell installs PySpark when it is missing and starts a local session that also writes a Spark event log.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=4.0,<4.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

# Python workers must use this notebook's interpreter, and every timestamp in the lessons is UTC.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["TZ"] = "UTC"
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-lessons-")
EVENT_LOG_DIR = os.path.join(WORK_DIR, "event-log")  # read back in the monitoring section
os.makedirs(EVENT_LOG_DIR)

spark = (
    SparkSession.builder
    .appName("pyspark-lessons")
    .master("local[2]")  # driver and executor share one JVM: the development topology
    .config("spark.sql.shuffle.partitions", "4")  # 200 is the default; tiny data needs few
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", EVENT_LOG_DIR)
    .config("spark.eventLog.compress", "false")  # plain JSON lines, readable below
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

## Sample data: the tables every lesson below runs on

This section seeds a small four-table slice of an orders platform, built with
`spark.createDataFrame` and an explicit schema per table. Three quirks the lessons lean on:
customer 4 has no `region`, customers 7 and 8 have never ordered, and one event is an exact
retry of the same row.

| table | grain | notable columns |
| --- | --- | --- |
| `customers` | one row per customer | `region`, `signup_date` (`DATE`) |
| `orders` | one row per order | `status`, `order_ts` (`TIMESTAMP`), `amount` (decimal) |
| `order_items` | one row per order line | `qty`, `unit_price` |
| `events` | one row per user event | `user_id`, `event`, `ts` |

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import types as T

customers_schema = T.StructType([
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("name", T.StringType(), False),
    T.StructField("region", T.StringType(), True),
    T.StructField("signup_date", T.DateType(), False),
])

customers = spark.createDataFrame(
    [
        (1, "Acme Corp",      "North", date(2024, 1, 3)),
        (2, "Bright Retail",  "North", date(2024, 2, 11)),
        (3, "Cyan Traders",   "South", date(2024, 3, 15)),
        (4, "Delta Grocers",  None,    date(2024, 5, 2)),
        (5, "Echo Supplies",  "West",  date(2024, 6, 20)),
        (6, "Foxtrot Foods",  "West",  date(2024, 8, 1)),
        (7, "Gamma Sports",   "East",  date(2024, 9, 14)),
        (8, "Hotel Homeware", "East",  date(2024, 11, 5)),
    ],
    schema=customers_schema,
)

customers.orderBy("customer_id").show()

In [ ]:
from pyspark.sql import types as T

orders_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("status", T.StringType(), True),
    T.StructField("order_ts", T.TimestampType(), False),
    T.StructField("amount", T.DecimalType(12, 2), True),
])

orders = spark.createDataFrame(
    [
        (1,  1, "paid",     datetime(2026, 1,  5,  9, 15), D("120.00")),
        (2,  1, "shipped",  datetime(2026, 1,  5, 10,  5),  D("84.50")),
        (3,  1, "returned", datetime(2026, 1,  5, 11, 40),  D("30.00")),
        (4,  2, "paid",     datetime(2026, 1,  6, 15, 20), D("260.00")),
        (5,  2, "placed",   datetime(2026, 1,  7,  8,  5),  D("55.25")),
        (6,  3, "shipped",  datetime(2026, 1,  7, 17, 45), D("149.00")),
        (7,  3, "paid",     datetime(2026, 1,  9, 12, 10), D("310.00")),
        (8,  4, "returned", datetime(2026, 1, 10, 19, 30),  D("89.90")),
        (9,  4, "shipped",  datetime(2026, 1, 12,  8, 55), D("1180.00")),
        (10, 5, "paid",     datetime(2026, 1, 12, 13,  0),   D("74.00")),
        (11, 6, "placed",   datetime(2026, 1, 13, 10, 25),  D("210.00")),
        (12, 6, "shipped",  datetime(2026, 1, 14,  0, 30),   D("96.40")),
    ],
    schema=orders_schema,
)

orders.orderBy("order_id").show()
orders.printSchema()

In [ ]:
from pyspark.sql import types as T

order_items_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("sku", T.StringType(), False),
    T.StructField("qty", T.IntegerType(), False),
    T.StructField("unit_price", T.DecimalType(8, 2), False),
])

order_items = spark.createDataFrame(
    [
        (1,  "SKU-4001",  2,  D("12.50")),
        (1,  "SKU-4007",  1,  D("95.00")),
        (2,  "SKU-4002",  1,  D("34.50")),
        (2,  "SKU-4009",  2,  D("25.00")),
        (3,  "SKU-4002",  1,  D("30.00")),
        (4,  "SKU-4003",  1, D("149.00")),
        (4,  "SKU-4004",  2,  D("55.50")),
        (5,  "SKU-4005",  1,  D("55.25")),
        (6,  "SKU-4003",  1, D("149.00")),
        (7,  "SKU-4006",  1, D("189.00")),
        (7,  "SKU-4008",  1, D("121.00")),
        (8,  "SKU-4010",  1,  D("89.90")),
        (9,  "SKU-4011", 20,  D("59.00")),
        (10, "SKU-4012",  2,  D("37.00")),
        (11, "SKU-4013",  1, D("210.00")),
        (12, "SKU-4014",  1,  D("41.40")),
        (12, "SKU-4005",  1,  D("55.00")),
    ],
    schema=order_items_schema,
)

order_items.orderBy("order_id", "sku").show(20)

In [ ]:
from pyspark.sql import types as T

events_schema = T.StructType([
    T.StructField("user_id", T.IntegerType(), False),
    T.StructField("event", T.StringType(), False),
    T.StructField("ts", T.TimestampType(), False),
])

events = spark.createDataFrame(
    [
        (1, "view",        datetime(2026, 1,  6,  9,  0)),
        (1, "add_to_cart", datetime(2026, 1,  6,  9,  2)),
        (1, "purchase",    datetime(2026, 1,  6,  9, 10)),
        (2, "view",        datetime(2026, 1,  7, 10,  0)),
        (2, "purchase",    datetime(2026, 1,  7, 10, 40)),
        (3, "view",        datetime(2026, 1,  6, 11, 15)),
        (3, "view",        datetime(2026, 1,  8, 11, 20)),
        (4, "add_to_cart", datetime(2026, 1,  9, 20,  5)),
        (4, "purchase",    datetime(2026, 1, 10,  8, 30)),
        (5, "view",        datetime(2026, 1, 11, 14, 45)),
        (5, "view",        datetime(2026, 1, 12,  9,  0)),
        (5, "purchase",    datetime(2026, 1, 12, 12, 30)),
        (5, "view",        datetime(2026, 1, 12,  9,  0)),
    ],
    schema=events_schema,
)

events.orderBy("user_id", "ts").show()

In [ ]:
from pyspark.sql import functions as F

frames = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "events": events,
}
expected_rows = {"customers": 8, "orders": 12, "order_items": 17, "events": 13}

for table, frame in frames.items():
    actual = frame.count()
    assert actual == expected_rows[table], (table, actual, expected_rows[table])
    print(f"{table:<12} {actual:>2} rows")

per_order = (
    order_items
    .groupBy("order_id")
    .agg(F.sum(F.col("qty") * F.col("unit_price")).alias("lines"))
    .join(orders.select("order_id", "amount"), "order_id")
)
assert per_order.where(F.col("lines") != F.col("amount")).count() == 0
print("line totals reconcile with the orders ledger:", float(orders.agg(F.sum("amount")).first()[0]))

assert events.dropDuplicates().count() == events.count() - 1
print("and the retried event collapses to 12 distinct rows - seed verified")

In [ ]:
import contextlib
import io

from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints, so cells can assert on plans."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()

### A larger, skewed orders table

The seed tables are too small to show a shuffle or a straggler, so the tuning lessons use a second pair. `big_orders` has 200,000 rows; customer `0` owns 40 percent of them - the "guest checkout" key that makes real joins skew. `big_customers` is a 5,000-row dimension. The cell also resets the session to a static baseline (16 shuffle partitions, AQE off), so each lever can be switched on deliberately.

In [ ]:
# Tuning lessons start from a static baseline; later cells switch levers on one at a time.
spark.conf.set("spark.sql.shuffle.partitions", "16")
spark.conf.set("spark.sql.adaptive.enabled", "false")

from pyspark.sql.functions import spark_partition_id


def partition_sizes(df):
    """Row count per partition, in partition order: the skew signal the Stages tab shows."""
    return [r["count"] for r in df.groupBy(spark_partition_id().alias("p")).count().orderBy("p").collect()]


ROWS, CUSTOMERS, HOT_SHARE = 200_000, 5_000, 0.4
big_orders = spark.range(ROWS).select(
    F.col("id").alias("order_id"),
    F.when(F.col("id") % 10 < HOT_SHARE * 10, F.lit(0)).otherwise(F.col("id") % CUSTOMERS).alias("customer_id"),
    F.date_add(F.lit("2026-10-01").cast("date"), (F.col("id") % 30).cast("int")).alias("order_date"),
    (F.col("id") % 997 / 10).cast("decimal(12,2)").alias("amount"),
    F.element_at(F.array(F.lit("paid"), F.lit("shipped"), F.lit("placed")), (F.col("id") % 3 + 1).cast("int")).alias("status"),
)
big_customers = spark.range(CUSTOMERS).select(
    F.col("id").alias("customer_id"),
    F.concat(F.lit("segment-"), (F.col("id") % 4).cast("string")).alias("customer_segment"),
)

# A partitioned Parquet copy, so scans, pruning and pushdown behave like a real table
ORDERS_PATH = os.path.join(WORK_DIR, "big_orders")
big_orders.write.mode("overwrite").partitionBy("order_date").parquet(ORDERS_PATH)
big_orders = spark.read.parquet(ORDERS_PATH)

hot = big_orders.where("customer_id = 0").count()
print(f"rows: {big_orders.count():,}  hot-key rows: {hot:,}")
assert hot == ROWS * HOT_SHARE

## How do you read a Spark physical plan?

In [ ]:
result = (
    big_orders
    .where(F.col("order_date") >= F.lit("2026-10-20"))
    .join(big_customers, "customer_id")
    .groupBy("customer_segment")
    .agg(F.sum("amount").alias("revenue"))
)
plan = plan_of(result)
print(plan)
assert "PartitionFilters" in plan and "BroadcastHashJoin" in plan and "Exchange hashpartitioning(customer_segment" in plan_of(result, "simple")

Read it bottom-up and look for:

+ `Scan parquet` with `ReadSchema` - which columns are read
+ `PartitionFilters` - directories pruned at plan time
+ `PushedFilters` - predicates the file reader evaluates
+ `Exchange` - a shuffle, i.e. a stage boundary
+ `BroadcastHashJoin` versus `SortMergeJoin` - whether the join shuffled both sides
+ `AdaptiveSparkPlan isFinalPlan=false` - AQE may still rewrite the plan at runtime

Catalyst runs analysis (resolve names and types), logical optimization (pushdown, constant
folding, column pruning), physical planning (pick the cheapest candidate from statistics), and
whole-stage code generation (fuse operators into one generated function, shown as
`WholeStageCodegen` or `*`). `explain("cost")` prints the optimized logical plan with the size
and row estimates the planner used - the way to see why it chose one join over another.

Common mistake: treating the printed root as the expensive part. Cost sits in the leaves (scans),
the exchanges, and the joins. And a pre-execution AQE plan is a guess; confirm with the final
plan or the UI stage metrics.

In [ ]:
cost_plan = plan_of(result, "cost")
print(cost_plan[:1500])
assert "Statistics(sizeInBytes=" in cost_plan

## How do you monitor and debug PySpark jobs?

Start with the Spark UI. On a cluster it lives on the driver (port 4040 while the application runs) or in the history server once it has finished. It shows jobs, stages, tasks, shuffle read and write, spill, input size, executor failures, and SQL plans annotated with runtime row counts.

In this notebook the UI is switched off (`spark.ui.enabled=false`) to keep a local or Colab run quiet and portable, so the cells read the same evidence from `explain`, partition counts and the event log. When the UI is on, look at:

+ **Stages:** sort by duration, open the slowest stage, compare the max task duration to the median, and read shuffle read/write and spill.
+ **SQL / DataFrame:** the executed plan with row counts per operator, and whether AQE changed the join or partition count.
+ **Executors:** GC time, failed tasks, lost executors, and how evenly work is spread.
+ **Storage:** which cached frames are actually materialized and how much memory they hold.

In [ ]:
from pyspark.sql import functions as F

enriched = (
    orders.alias("o")
    .join(customers.alias("c"), on="customer_id", how="left")
    .select(
        "o.order_id",
        "o.customer_id",
        "c.name",
        "c.region",
        "o.status",
        "o.amount",
    )
    .orderBy("o.order_id")
)
enriched.show()

# `customer_id` is unique on the dimension, so the enrich join must not fan out
assert enriched.count() == orders.count()
print("join verified: one row per order")

Useful PySpark checks:

In [ ]:
print(plan_of(enriched, "simple"))
print("partitions:", enriched.rdd.getNumPartitions())
print("shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

How it works under the hood. The driver's DAG scheduler tracks stages and tasks and emits the event log the Spark UI reads. The numbers that matter are per-stage: shuffle read and write bytes, spill to memory and disk, the task duration distribution, and GC time. A stage boundary exists wherever the plan contains an `Exchange`.

### Diagnose a slow stage from its metrics

How to read the stage numbers: if the median task takes 20 seconds and the maximum
12 minutes, the stage is skewed and more executors will not help; if every task takes minutes
and spills, partitions are too large; 50,000 millisecond tasks mean over-partitioning. The SQL
tab shows the plan annotated with runtime row counts.

| Metric you see | Likely cause | Fix |
| --- | --- | --- |
| Task duration max much greater than median | Skew or one oversized partition | Salt the hot key, repartition, enable AQE skew join |
| Shuffle read or write bytes large | A wide transformation | Broadcast the small side, pre-aggregate, filter earlier, change the key |
| Spill (memory) and Spill (disk) above zero | Partition does not fit execution memory | More shuffle partitions, smaller advisory size, more executor memory |
| GC time a large fraction of task time | Too many Java objects | Kryo, serialized persist levels, fewer cached rows |
| Many tiny tasks, CPU mostly idle | Over-partitioning | Reduce partitions, let AQE coalesce |
| Input tasks tiny while listing is slow | Small files | Compact with `OPTIMIZE`, tune write layout |
| Executor lost or container killed | Off-heap or overhead exhaustion | Raise `spark.executor.memoryOverhead`, lower cores per executor |
| Driver out-of-memory | `collect`, `toPandas`, or a large broadcast | Remove the collection, lower the broadcast threshold |
| Fetch failures and re-run map stages | Shuffle files lost with an executor | Shuffle service, fewer lost executors (spot), retries |

`explain` output is a plan, not a runtime fact; only the UI or the event log are. Production
jobs should set `spark.eventLog.enabled=true` and `spark.eventLog.dir` for a history server. Event
logs are newline-delimited JSON, so you can read per-stage metrics directly - here from this
session's own log:

In [ ]:
import glob
import json
import os
from collections import defaultdict


def event_log_files(log_path):
    """Spark 3 writes one event log file; Spark 4 writes a directory of rolling event files."""
    if os.path.isdir(log_path):
        return sorted(glob.glob(os.path.join(log_path, "events_*")))
    return [log_path]


def stage_metrics(log_path):
    """Sum tasks, shuffle and spill bytes per stage from a Spark event log."""
    totals = defaultdict(lambda: {"tasks": 0, "shuffle_read": 0, "shuffle_write": 0, "spill_disk": 0, "spill_mem": 0})
    for path in event_log_files(log_path):
        with open(path) as handle:
            for line in handle:
                event = json.loads(line)
                if event.get("Event") != "SparkListenerTaskEnd":
                    continue
                metrics = event.get("Task Metrics") or {}
                read = metrics.get("Shuffle Read Metrics") or {}
                write = metrics.get("Shuffle Write Metrics") or {}
                stage = totals[event["Stage ID"]]
                stage["tasks"] += 1
                stage["shuffle_read"] += read.get("Remote Bytes Read", 0) + read.get("Local Bytes Read", 0)
                stage["shuffle_write"] += write.get("Shuffle Bytes Written", 0)
                stage["spill_disk"] += metrics.get("Disk Bytes Spilled", 0)
                stage["spill_mem"] += metrics.get("Memory Bytes Spilled", 0)
    return dict(totals)


spark.conf.set("spark.sql.adaptive.enabled", "false")
big_orders.groupBy("customer_id").agg(F.sum("amount")).collect()  # one shuffle to measure

(log_path,) = glob.glob(os.path.join(EVENT_LOG_DIR, "*"))
metrics = stage_metrics(log_path)
heaviest = max(metrics.items(), key=lambda item: item[1]["shuffle_write"])
print(f"{len(metrics)} stages in the log; heaviest shuffle writer: stage {heaviest[0]} {heaviest[1]}")
assert heaviest[1]["shuffle_write"] > 0

Common mistake: seeing a slow stage and raising executor memory. Spill points at memory or
partition size, skew at the key, a large shuffle at the plan, tiny tasks at partitioning -
diagnose the class first, then pick the lever.

### Log record counts at each pipeline boundary

Row counts are the cheapest monitoring signal: a stage that suddenly drops or multiplies rows is a correctness bug before it is a performance one. The quality gate below caches the checked frame because both writes read it, then logs input and output counts to a metrics table:

In [ ]:
bronze_orders_raw = spark.createDataFrame(
    [(1, 10, D("50.00")), (2, None, D("20.00")), (None, 11, D("5.00")), (4, 12, D("-3.00")), (5, 13, D("8.00"))],
    "order_id int, customer_id int, amount decimal(12,2)",
)

checked = bronze_orders_raw.withColumn(
    "quality_error",
    F.when(F.col("order_id").isNull(), "missing_order_id")
     .when(F.col("customer_id").isNull(), "missing_customer_id")
     .when(F.col("amount") < 0, "negative_amount"),
).cache()  # both writes below read the same checked frame

valid_orders = checked.where("quality_error is null").drop("quality_error")
quarantine = checked.where("quality_error is not null").withColumn("rejected_at", F.current_timestamp())

valid_orders.write.mode("overwrite").saveAsTable("silver_orders_checked")
quarantine.write.mode("overwrite").saveAsTable("ops_orders_quarantine")

quarantine.groupBy("quality_error").count().orderBy("quality_error").show()
assert valid_orders.count() == 2 and quarantine.count() == 3
checked.unpersist()

In [ ]:
input_count = bronze_orders_raw.count()
output_count = valid_orders.count()

spark.createDataFrame(
    [("orders_silver", input_count, output_count)],
    "pipeline_name string, input_count long, output_count long",
).write.mode("append").saveAsTable("ops_pipeline_metrics")

spark.table("ops_pipeline_metrics").show()
assert (input_count, output_count) == (5, 2)

## How should you approach tuning, and how do you measure a change?

Start with evidence, not configuration:

1. Reproduce the slow job with representative data.
2. Read the Spark UI and `explain("formatted")`.
3. Identify the bottleneck: scan, shuffle, skew, spill, join, small files, driver memory, or sink write.
4. Change one thing.
5. Measure again.

Spark turns a DataFrame program into a logical plan, Catalyst optimizes it, and the physical
plan becomes a DAG of **stages**, cut wherever data is exchanged across the network. Inside a
stage Spark runs one task per partition, and the stage finishes only when its slowest task
does. A stage with 1,000 balanced tasks and one straggler is as slow as the straggler, no matter
how much memory you add.

Worked example: a job reads 4 TB, filters to 200 GB, and groups by `order_date`. Scan
parallelism is set by the file layout; shuffle parallelism by `spark.sql.shuffle.partitions`
and AQE. With the default 200 shuffle partitions each reducer handles about 1 GB and spills;
2,000 partitions bring it to about 100 MB.

Common mistake: reaching for executor memory first. More memory helps only when the UI shows
spill, high GC time, or killed containers - not when one task dominates or partitions are never
pruned.

Treat it as an experiment on the same data, code, and cluster. Before and after, record wall
clock, shuffle read and write, spill, task count and max task duration, GC time, output file
count, and cost per run. Shuffle bytes are the leading indicator; wall clock is what the business
sees but also the noisiest, so compare medians over several runs. Cost is a separate axis: halving
runtime on a cluster twice the size buys latency, not efficiency. Change one thing at a time and
check that the output is identical.

In [ ]:
import json

# Record the same fields before and after, then compare like with like.
def summarize(before, after):
    def delta(key):
        b, a = before[key], after[key]
        change = (a - b) / b * 100 if b else 0.0
        return f"{b:,} -> {a:,} ({change:+.1f}%)"

    for key in ("wall_clock_s", "shuffle_read_bytes", "shuffle_write_bytes",
                "disk_bytes_spilled", "max_task_s", "task_count"):
        print(f"{key:22s} {delta(key)}")


baseline = {"wall_clock_s": 3600, "shuffle_read_bytes": 1_200_000_000_000,
            "shuffle_write_bytes": 1_200_000_000_000, "disk_bytes_spilled": 8_000_000_000,
            "max_task_s": 540, "task_count": 200_000}
tuned = {"wall_clock_s": 1_200, "shuffle_read_bytes": 20_000_000_000,
         "shuffle_write_bytes": 20_000_000_000, "disk_bytes_spilled": 0,
         "max_task_s": 95, "task_count": 40_000}

summarize(baseline, tuned)
print(json.dumps({"shuffle_reduction_pct": round(
    (1 - tuned["shuffle_read_bytes"] / baseline["shuffle_read_bytes"]) * 100, 1)}))

## What is a shuffle and why is it expensive?

A shuffle redistributes rows so that all rows with the same key land in the same partition.
`groupBy`, `distinct`, `dropDuplicates`, non-broadcast joins, `repartition`, `orderBy`, and
window functions with `partitionBy` all shuffle.

Each map task partitions its output by `hash(key) % numPartitions`, sorts it, and writes one
block per reducer to local disk; reducers fetch their blocks from every mapper over the network.
That costs serialization, a disk write and read, network transfer, and a sort or hash build.
Fetch failures force map outputs to be recomputed, so a failed shuffle stage can cost far more
than its first attempt.

Worked example: moving 500 GB at an aggregate 1 GB/s costs at least 500 seconds of transfer
before serialization and sorting. Broadcasting a 20 MB dimension instead removes the shuffle.

Common mistake: assuming short code is cheap. `orderBy` before a write is a range shuffle,
`countDistinct` on a high-cardinality column shuffles every distinct value, and `repartition()`
with no arguments is a full shuffle for no benefit. Count the exchanges in the plan:

In [ ]:
daily_revenue = big_orders.groupBy("order_date").agg(F.sum("amount").alias("revenue"))
ordered = daily_revenue.orderBy("order_date")

for name, frame in [("groupBy", daily_revenue), ("groupBy + orderBy", ordered)]:
    print(f"{name:<18} exchanges: {plan_of(frame, 'simple').count('Exchange ')}")
assert plan_of(daily_revenue, "simple").count("Exchange ") == 1
assert plan_of(ordered, "simple").count("Exchange ") == 2  # the sort adds a range-partitioned shuffle

## How do you choose the number of shuffle partitions?

`spark.sql.shuffle.partitions` (default 200) sets the reduce fan-out of every shuffle. Too few
partitions mean large tasks that spill; too many mean scheduling overhead and tiny output files.
Derive it from a target partition size of roughly 100-200 MB:
`partitions = ceil(shuffle_bytes / target_size)`.

With AQE on, the setting is only the starting count:
`spark.sql.adaptive.advisoryPartitionSizeInBytes` (default 64 MB) is the size AQE coalesces toward
once it sees the real map output. A 2 TB shuffle at 64 MB implies about 32,000 partitions, at
128 MB about 16,000.

Common mistakes: a huge round number "just in case" with AQE off (tens of thousands of tiny
tasks and files), and changing the setting after the plan was built - it applies to the next
action only. On tiny data, AQE collapses the 16 configured partitions:

In [ ]:
by_customer = lambda: big_orders.groupBy("customer_id").agg(F.sum("amount").alias("revenue"))  # noqa: E731

spark.conf.set("spark.sql.adaptive.enabled", "false")
static_partitions = by_customer().rdd.getNumPartitions()

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
adaptive_partitions = by_customer().rdd.getNumPartitions()

print("AQE off:", static_partitions, "partitions | AQE on:", adaptive_partitions)
assert static_partitions == 16 and adaptive_partitions < static_partitions

In [ ]:
import math


def shuffle_partitions_for(shuffle_bytes, target_bytes=128 * 1024**2):
    return math.ceil(shuffle_bytes / target_bytes)


for label, size in [("200 GB", 200 * 1024**3), ("2 TB", 2 * 1024**4)]:
    print(f"{label:>6} shuffle -> {shuffle_partitions_for(size):,} partitions at 128 MB")
assert shuffle_partitions_for(2 * 1024**4, 64 * 1024**2) == 32_768

## How does Adaptive Query Execution help?

AQE (on by default since Spark 3.2) re-plans between query stages. When the map side of a
shuffle finishes, Spark knows the real partition sizes and can:

+ coalesce many small post-shuffle partitions (`spark.sql.adaptive.coalescePartitions.enabled`),
+ split skewed join partitions (`spark.sql.adaptive.skewJoin.enabled`) - a partition counts as
  skewed above both `skewedPartitionThresholdInBytes` (256 MB) and `skewedPartitionFactor` (5) times the median,
+ switch a sort-merge join to a broadcast join when a side turns out small,
+ read shuffle files locally (`spark.sql.adaptive.localShuffleReader.enabled`).

Worked example: 200 shuffle partitions where 190 are 5 MB and ten are 2 GB. Without AQE you pay
200 tasks and ten that spill; with AQE the small ones coalesce toward 64 MB and the large ones
split.

Common mistake: expecting AQE to fix everything. It acts only at shuffle boundaries, so it cannot
fix a skewed narrow stage, a bad partition key, or a scan that reads every file. Here the
optimizer's estimate rules out a broadcast, and AQE switches once it sees the filtered size:

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(64 * 1024))  # 64 KB: too small for the raw scan

tiny_customers = big_customers.where("customer_id < 50")  # the planner cannot know this is tiny
joined = big_orders.join(tiny_customers.groupBy("customer_id").agg(F.max("customer_segment").alias("seg")), "customer_id")
before = plan_of(joined)
joined.collect()  # run it so AQE can re-plan with real sizes
final = joined._jdf.queryExecution().executedPlan().toString()  # the plan that actually ran (internal API)
print("initial plan:", "SortMergeJoin" if "SortMergeJoin" in before else "?",
      "| final plan:", "BroadcastHashJoin" if "BroadcastHashJoin" in final else "?")

spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
assert "SortMergeJoin" in before and "isFinalPlan=false" in before
assert "isFinalPlan=true" in final and "BroadcastHashJoin" in final

## How do you fix data skew with salting?

When a few keys hold most rows, one task owns most of the work. Salting adds a random bucket to
the hot side and replicates the small side across all buckets, so the hot key spreads over many
tasks. Use it when broadcasting is impossible and AQE skew handling is not enough - and only
after the UI or a key count confirmed the skew.

In [ ]:
salt_buckets = 16

orders_salted = big_orders.withColumn("salt", (F.rand(seed=42) * salt_buckets).cast("int"))
customers_salted = big_customers.crossJoin(spark.range(salt_buckets).withColumnRenamed("id", "salt"))

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # force a shuffle join for the demo
plain = big_orders.join(big_customers, "customer_id")
salted = orders_salted.join(customers_salted, ["customer_id", "salt"]).drop("salt")

hot_before = max(partition_sizes(big_orders.repartition(16, "customer_id")))
hot_after = max(partition_sizes(orders_salted.repartition(16, "customer_id", "salt")))
print(f"largest join partition: {hot_before:,} rows before salting, {hot_after:,} after")

assert salted.count() == plain.count() == ROWS  # salting moves rows, it never changes matches
assert hot_after < hot_before / 2
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

Aggregations salt in two phases: aggregate by `(key, salt)`, then by `key`:

In [ ]:
partial = orders_salted.groupBy("customer_id", "salt").agg(F.sum("amount").alias("partial_amount"))
final = partial.groupBy("customer_id").agg(F.sum("partial_amount").alias("total_amount"))

direct = big_orders.groupBy("customer_id").agg(F.sum("amount").alias("total_amount"))
assert final.exceptAll(direct).count() == 0
print("two-phase salted aggregate matches the direct aggregate")

Worked example: one billion rows with one key holding 40 percent is 400 million rows in one
task. With 32 salts each bucket holds 12.5 million, but the small side is replicated 32 times,
so a 20 MB dimension becomes 640 MB and may no longer fit the broadcast budget.

Common mistakes: salting without confirmed skew (an extra shuffle for nothing), forgetting to
drop the salt column, and salting only one side so keys never match. A salt computed from a hash
of the key alone does not help: every row of the hot key still gets the same salt.

## How do you tune joins?

Start with the size and distribution of both sides: broadcast a small side, filter and select
before joining, deduplicate dimensions so facts are not multiplied, and handle hot keys.

+ `BroadcastHashJoin` ships the small side to every executor; the large side never shuffles.
+ `SortMergeJoin` shuffles and sorts both sides; it scales to two large inputs and spills gracefully.
+ `ShuffledHashJoin` shuffles both sides but hash-builds instead of sorting
  (`spark.sql.join.preferSortMergeJoin=false`, or chosen by AQE).
+ Non-equi joins fall back to `BroadcastNestedLoopJoin` or a Cartesian product - almost always wrong.

Common mistakes: joining before filtering; joining on an expression such as
`lower(a.id) == lower(b.id)` instead of a clean key column; and a dimension that is not unique on
the key, which silently doubles revenue:

In [ ]:
dim_with_history = big_customers.unionByName(big_customers.where("customer_id < 10"))  # two versions of 10 rows

fanned_out = big_orders.join(dim_with_history, "customer_id").count()
deduped = big_orders.join(dim_with_history.dropDuplicates(["customer_id"]), "customer_id").count()
print(f"rows after join: {fanned_out:,} with a non-unique dimension, {deduped:,} after dedupe")
assert deduped == ROWS and fanned_out > ROWS

## How does broadcast join tuning work?

Spark broadcasts automatically when the estimated size of a side is below
`spark.sql.autoBroadcastJoinThreshold` (10 MB; `-1` disables it); `broadcast()` overrides the
estimate. At runtime the small side is collected to the driver and served to executors as
torrent blocks, so the costs are driver memory, executor memory, and broadcast latency.

Worked example: a 10-million-row product dimension at about 100 bytes per row is 1 GB - 100x the
default threshold. Forcing a broadcast because the table is called "dim" makes the driver and
every executor hold 1 GB, and the job fails instead of running slowly.

Common mistakes: forcing a broadcast without checking the size, raising the threshold globally to
make one join broadcast, and broadcasting the result of an expensive subquery. Validate in the
plan:

In [ ]:
from pyspark.sql.functions import broadcast

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
shuffled_plan = plan_of(big_orders.join(big_customers, "customer_id"))
hinted_plan = plan_of(big_orders.join(broadcast(big_customers), "customer_id"))
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

assert "SortMergeJoin" in shuffled_plan
assert "BroadcastHashJoin" in hinted_plan and "BroadcastExchange" in hinted_plan
print("threshold off -> SortMergeJoin; broadcast() hint -> BroadcastHashJoin")

## How do repartition and coalesce differ?

`repartition(n)` is a full shuffle (round-robin, or hash when you pass columns); it can raise or
lower the count and balances by construction. `coalesce(n)` merges existing partitions without a
shuffle (a narrow dependency), can only lower the count, and keeps any upstream imbalance because
it never splits a partition.

Worked example: after a filter, 1,000 partitions hold 4 GB. `coalesce(16)` gives tasks of about
256 MB with no shuffle - likely spilling; `repartition(16)` pays one shuffle for 16 even
partitions.

Common mistake: `coalesce(1)` on a large result "to avoid small files", which funnels the whole
write onto one core.

In [ ]:
skewed = big_orders.repartition(8, "customer_id")  # the hot key lands in a single partition
merged = skewed.coalesce(4)  # narrow: merges whole partitions, keeps the hot one intact
balanced = skewed.repartition(4)  # full round-robin shuffle

for name, frame in [("hash by customer", skewed), ("coalesce(4)", merged), ("repartition(4)", balanced)]:
    print(f"{name:<17} {partition_sizes(frame)}")

exchanges = lambda df: plan_of(df, "simple").count("Exchange ")  # noqa: E731
assert exchanges(merged) == exchanges(skewed)  # coalesce adds no shuffle
assert max(partition_sizes(merged)) >= ROWS * HOT_SHARE
assert max(partition_sizes(balanced)) - min(partition_sizes(balanced)) < ROWS * 0.05

## How do you use repartition by column?

`repartition(n, col)` hash-partitions: `partition = pmod(hash(value), n)`, so equal keys always
meet - what a keyed join, aggregation, or dedup needs. Hashing destroys ordering, and all nulls
hash to one partition, so a large null group is a skew problem. The cell above shows the other
trap: one hot value makes one partition huge whatever `n` is.

Before a partitioned write, `repartition("order_date")` followed by `partitionBy("order_date")`
writes roughly one file per date; skipping it writes one file per (task, date) pair:

In [ ]:
def file_count(path):
    return sum(name.endswith(".parquet") for _, _, names in os.walk(path) for name in names)


scattered_path = os.path.join(WORK_DIR, "scattered")
clustered_path = os.path.join(WORK_DIR, "clustered")
big_orders.repartition(8).write.mode("overwrite").partitionBy("order_date").parquet(scattered_path)
big_orders.repartition("order_date").write.mode("overwrite").partitionBy("order_date").parquet(clustered_path)

print("files without repartition by date:", file_count(scattered_path))
print("files with repartition by date   :", file_count(clustered_path))
assert file_count(clustered_path) == 30 and file_count(scattered_path) == 30 * 8

Common mistakes: repartitioning by one key and immediately shuffling on another (two exchanges
where one was needed), and repartitioning by a low-cardinality column, which leaves most
partitions empty while the largest value dominates.

## How do you tune file sizes and avoid small files?

Spark packs files into scan tasks using `spark.sql.files.maxPartitionBytes` (128 MB) and
`spark.sql.files.openCostInBytes` (4 MB, the modelled cost of opening a file). With tiny files the
task count stays reasonable but listing, opening, and footer reads dominate, and table formats
also carry more per-file metadata, so planning slows down.

Worked example: 500,000 files of 1 MB (500 GB) compacted to 1 GB files leaves about 500 files -
1,000x fewer opens for the same bytes. Most analytical tables settle on 128 MB-1 GB per file.

Control the output file count before writing, and compact what already exists:

In [ ]:
small_path = os.path.join(WORK_DIR, "small_files")
compact_path = os.path.join(WORK_DIR, "compacted")
day = big_orders.where("order_date = date '2026-10-05'")

day.repartition(24).write.mode("overwrite").parquet(small_path)
spark.read.parquet(small_path).coalesce(1).write.mode("overwrite").parquet(compact_path)

print("files before compaction:", file_count(small_path), "| after:", file_count(compact_path))
assert file_count(small_path) == 24 and file_count(compact_path) == 1
assert spark.read.parquet(compact_path).count() == day.count()

On table formats use the platform's compaction instead of rewriting files yourself:

```sql
-- Delta
OPTIMIZE main.silver.orders WHERE order_date >= date '2026-10-01';
-- Iceberg (Spark procedure)
CALL catalog.system.rewrite_data_files(table => 'silver.orders');
```

Common mistakes: `coalesce(1)` on a large table (one core does the write - fine above only because
one day is tiny), rewriting a whole table to fix one partition, and leaving write-time compaction
(`delta.autoOptimize.optimizeWrite`, `delta.autoOptimize.autoCompact`) off.

## How do you tune partition pruning and predicate pushdown?

Two mechanisms. **Partition pruning** uses the directory layout: Spark evaluates the predicate
on the partition values and lists the survivors under `PartitionFilters` - even an expression on
the partition column can be evaluated this way. **Pushdown and data skipping** work inside files:
Parquet row-group min/max and Delta per-file statistics let the reader skip data, and pushed
predicates appear under `PushedFilters`. Wrapping a regular column in a function hides the
literal from the reader:

In [ ]:
def plan_line(df, prefix):
    return next(line.strip() for line in plan_of(df).splitlines() if line.strip().startswith(prefix))


raw_predicate = big_orders.where("status = 'paid'")
wrapped_predicate = big_orders.where("lower(status) = 'paid'")
partition_expression = big_orders.where("date_format(order_date, 'yyyy-MM-dd') = '2026-10-05'")

print(plan_line(raw_predicate, "PushedFilters"))
print(plan_line(wrapped_predicate, "PushedFilters"))
print(plan_line(partition_expression, "PartitionFilters"))

assert "EqualTo(status,paid)" in plan_line(raw_predicate, "PushedFilters")
assert "EqualTo" not in plan_line(wrapped_predicate, "PushedFilters")
assert "date_format" in plan_line(partition_expression, "PartitionFilters")

Worked example: five years partitioned by day is about 1,825 directories, and a one-day query
reads about 1/1825 of the files. Inside a 128 MB Parquet file, well-clustered values let the
reader skip most row groups.

Common mistakes: `lower(country) = 'us'`, `cast(ts as string) like '2026-10%'`, `col like '%x'`,
or comparing a timestamp to a string literal - each defeats pushdown on a regular column. Delta
indexes statistics for the first 32 columns (`delta.dataSkippingNumIndexedCols`); for Iceberg use
hidden partition transforms such as `days(event_ts)` or `bucket(32, customer_id)`.

## How do you reduce data before expensive operations?

Filter rows and select columns before joins, aggregations, and windows. Catalyst pushes
deterministic filters and projections toward the scan anyway - both versions below get the same
pushed-down scan - but it cannot push through aggregations, windows, `distinct`, Python UDFs, or
non-deterministic expressions such as `rand()`, so writing the intent early keeps it safe:

In [ ]:
early = (
    big_orders.where("order_date >= date '2026-10-20'")
    .select("customer_id", "amount")
    .join(big_customers, "customer_id")
    .groupBy("customer_segment").agg(F.sum("amount").alias("revenue"))
)
late = (
    big_orders.join(big_customers, "customer_id")
    .where("order_date >= date '2026-10-20'")
    .groupBy("customer_segment").agg(F.sum("amount").alias("revenue"))
)
assert plan_line(early, "PartitionFilters") == plan_line(late, "PartitionFilters")
assert sorted(early.collect()) == sorted(late.collect())
print("Catalyst pushed the late filter down to the same scan:", plan_line(late, "PartitionFilters"))

Worked example: a 60-column, 2-billion-row table where the report needs four columns and one
month - selecting first reads 4 column chunks instead of 60, and the month filter cuts rows about
12x. Those two lines change the join input more than any executor setting.

## When should you cache or persist DataFrames?

Cache only a DataFrame that is reused, expensive to recompute, and fits. `cache()` and
`persist()` are lazy: blocks are stored on the first action. `MEMORY_ONLY` keeps deserialized
objects (fast, large; evicted partitions are recomputed), `MEMORY_AND_DISK` spills evicted blocks,
`_SER` levels store compact bytes at a CPU cost, `DISK_ONLY` always writes to disk. Storage shares
the unified memory region with execution, so a big shuffle can evict cached blocks.

Worked example: 100 GB cached across 20 executors with 16 GB heaps sounds like it fits 320 GB.
The unified region is 0.6 of (heap - 300 MB), about 188 GB, and storage holds half of that
without eviction - so the cache thrashes.

Common mistakes: caching by reflex, caching a frame read once, and forgetting `unpersist()`.

In [ ]:
from pyspark import StorageLevel

base = (
    big_orders
    .where("order_date >= date '2026-10-20'")
    .select("order_id", "customer_id", "amount", "order_date")
    .persist(StorageLevel.MEMORY_AND_DISK)
)
high_value = base.where("amount >= 90").count()  # the first action materializes the cache
daily = base.groupBy("order_date").agg(F.sum("amount").alias("revenue")).count()

print("storage level:", base.storageLevel, "| high-value big_orders:", high_value, "| days:", daily)
assert base.is_cached and daily == 11  # 20-30 October
base.unpersist()

## How do you avoid driver memory problems?

The driver holds the plan, scheduler state, broadcast values, and every collected result.
`collect()` and `toPandas()` pull the whole result into one JVM; `spark.driver.maxResultSize`
(1 GB) caps it, and the driver heap often fails first. A 50-million-row result at 200 bytes per
row is 10 GB. Aggregate in the cluster, write large results to tables, and collect only bounded
samples:

In [ ]:
summary = big_orders.agg(F.sum("amount").alias("total"), F.count("*").alias("rows")).first()
sample = big_orders.select("order_id", "customer_id").limit(5).collect()

print("aggregated in the cluster:", summary.asDict())
print("bounded sample:", [row["order_id"] for row in sample])
assert summary["rows"] == ROWS and len(sample) == 5

## How do you size executors, memory and parallelism?

Three independent numbers: cores per executor (task slots), memory per executor (shared by
those tasks), and executor count. Total parallelism is `executors x cores`, and shuffle
partitions should be a small multiple of that so the last wave does not leave cores idle.

An executor heap loses 300 MB reserved memory; the unified region is `spark.memory.fraction`
(0.6) of the rest, shared by execution (shuffles, sorts, joins) and storage (cache), with
`spark.memory.storageFraction` (0.5) protected for storage. `spark.executor.memoryOverhead`
(default `max(384 MB, 0.10 x heap)`) covers Python workers, Netty, and JVM internals, and the
container is heap plus overhead. Containers killed by the cluster manager usually mean
overhead, especially for Python-heavy jobs.

Three to five cores and 8-32 GB per executor is the usual landing zone: very wide executors
create long GC pauses, and many one-core executors multiply JVM overhead and broadcast copies.
Dynamic allocation (`spark.dynamicAllocation.*`, with `shuffleTracking.enabled`) suits bursty
shared jobs; static allocation suits stable nightly jobs. Raise `spark.driver.memory` alongside
any large broadcast. These are launch-time settings:

```python
spark = (
    SparkSession.builder
    .appName("large-orders-aggregation")
    .config("spark.executor.instances", "60")
    .config("spark.executor.cores", "5")
    .config("spark.executor.memory", "38g")
    .config("spark.executor.memoryOverhead", "4g")
    .config("spark.sql.shuffle.partitions", "1200")
    .getOrCreate()
)
```

The arithmetic behind those numbers:

In [ ]:
def executor_layout(heap_gb, cores, memory_fraction=0.6, reserved_gb=0.3):
    overhead_gb = max(0.384, 0.10 * heap_gb)
    unified_gb = memory_fraction * (heap_gb - reserved_gb)
    return {
        "container_gb": round(heap_gb + overhead_gb, 1),
        "unified_gb": round(unified_gb, 1),
        "execution_per_task_gb": round(unified_gb / cores, 2),
    }


layout = executor_layout(heap_gb=16, cores=5)
print(layout)
assert layout == {"container_gb": 17.6, "unified_gb": 9.4, "execution_per_task_gb": 1.88}

Worked example (cluster sizing): 10 nodes with 32 cores and 256 GB each; leave 1 core and 1 GB
per node for the OS, so 31 cores and 255 GB. Five cores per executor gives 6 executors per node,
60 in total, about 42 GB each - roughly 38 GB of heap once overhead is subtracted.

Common mistakes: sizing by total memory alone (one 32-core JVM per node), and forgetting that
overhead is added to the heap, so a container can fail to schedule though the heap would fit.

## How do you handle spilling and memory pressure?

When a sort, hash aggregate, or hash join exhausts its task's execution memory, Spark writes
sorted runs to local disk and merges them later - correct, but now disk-bound. Spill is per
task, so spill on a handful of tasks is itself a skew signal. The UI shows `Spill (Memory)` (bytes
held in memory before spilling) and `Spill (Disk)` (bytes written); the event log reports
`Memory Bytes Spilled` and `Disk Bytes Spilled`.

The cause is almost always partition size versus per-task memory: a 3 GB partition with about
1 GB per task spills. Fix it in this order: more shuffle partitions or a smaller advisory size;
filter and aggregate earlier; drop an unnecessary sort; broadcast the small side; only then add
executor memory. If one partition holds 10 percent of a 2 TB shuffle, no amount of memory helps -
the partition is the problem. Caching more while already spilling makes it worse.

```python
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")
spark.conf.set("spark.sql.shuffle.partitions", "8000")
```

## How do you tune serialization, caching and GC?

`spark.serializer=org.apache.spark.serializer.KryoSerializer` is faster and often 2-10x more
compact than Java serialization for RDD shuffles, broadcasts, and caches (DataFrames already use
Tungsten's binary format). With `spark.kryo.registrationRequired=true` every class must be
registered or the job fails; raise `spark.kryoserializer.buffer.max` on buffer-full errors.

A 30-byte logical record can occupy hundreds of bytes as deserialized JVM objects, which is why a
frame that does not fit `MEMORY_ONLY` often fits `MEMORY_AND_DISK_SER`. For GC, G1
(`-XX:+UseG1GC` in `spark.executor.extraJavaOptions`) is the usual choice; high GC time means too
many live objects - oversized caches, very wide rows, or giant executors. Off-heap storage
(`spark.memory.offHeap.enabled`, `spark.memory.offHeap.size`) removes cached bytes from GC.

Common mistakes: expecting a serializer change to speed up an I/O-bound scan or a skewed task,
using `_SER` levels for a small frame read repeatedly, and enabling `registrationRequired` before
registering classes.

## How do you tune Python UDFs and pandas UDFs?

A row-at-a-time Python UDF serializes every row to a Python worker and back, is opaque to
Catalyst (no pushdown, no constant folding), and breaks whole-stage code generation. Order of
preference: built-in expression, SQL expression, vectorized pandas UDF, row UDF. Most "UDFs"
already exist: `F.when`, `F.regexp_extract`, `F.xxhash64`, `F.date_add`. The plan shows the
boundary as `BatchEvalPython`:

In [ ]:
from pyspark.sql import types as T


@F.udf(T.StringType())
def tier_udf(amount):
    return "high" if amount is not None and amount >= 90 else "standard"


builtin_tier = big_orders.withColumn("tier", F.when(F.col("amount") >= 90, "high").otherwise("standard"))
udf_tier = big_orders.withColumn("tier", tier_udf("amount"))

assert "BatchEvalPython" in plan_of(udf_tier) and "BatchEvalPython" not in plan_of(builtin_tier)
assert builtin_tier.where("tier = 'high'").count() == udf_tier.where("tier = 'high'").count()
print("same result; only the UDF plan leaves the JVM")

Pandas UDFs amortize the boundary: Arrow batches (`spark.sql.execution.arrow.maxRecordsPerBatch`,
10,000 rows) arrive as `pandas.Series` and run vectorized. `mapInPandas` can change the row count;
`applyInPandas` runs once per group. They need `pandas` and `pyarrow` on every worker:

```python
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf("double")
def zscore(values: pd.Series) -> pd.Series:
    return (values - values.mean()) / values.std()

df = df.withColumn("amount_z", zscore("amount"))
```

On Databricks, Photon accelerates native operators but not Python UDFs, so a UDF in the hot path
loses the speedup twice. A pandas UDF is cheaper than a row UDF but still crosses the boundary,
so filter and aggregate in the JVM first.

## How do you tune Structured Streaming jobs?

A streaming query is a micro-batch loop: each trigger runs a normal job over newly available
data, and the checkpoint stores offsets, commits, and the state store. Bound each batch with
`maxOffsetsPerTrigger` (Kafka) or `maxFilesPerTrigger` (files), choose a trigger interval each
batch can finish within, bound state with watermarks, and keep checkpoint paths stable and one
per stream.

Worked example: with a 60-second trigger, batches start taking 90 seconds as the topic grows, so
each batch starts later with more data and latency climbs without bound. Capping
`maxOffsetsPerTrigger` and widening the trigger to 2 minutes stabilizes it.

```python
stream = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker1:9092")
    .option("subscribe", "orders")
    .option("maxOffsetsPerTrigger", 500_000)     # bound each micro-batch
    .load()
)
(
    stream.withWatermark("event_ts", "30 minutes")
    .dropDuplicatesWithinWatermark(["event_id"])
    .writeStream
    .option("checkpointLocation", "s3://acme-checkpoints/orders/")
    .trigger(processingTime="2 minutes")
    .toTable("main.silver.orders")
)
```

Common mistakes: stateful dedup without a watermark (state grows until the job slows down a week
later), moving the checkpoint (orphans offsets and state), and a full-table scan inside
`foreachBatch` that repeats every micro-batch.

## How do you tune Delta and file layout?

Layout changes the cost of every future query, not just one job. Aim for 128 MB-1 GB files;
`OPTIMIZE` compacts small files, `ZORDER BY (col)` co-locates values for data skipping (strongest
for the first column, degrading as new data arrives), and **liquid clustering**
(`CLUSTER BY (cols)`) records clustering keys in metadata and re-clusters incrementally. Liquid
clustering cannot be combined with `partitionBy` or `ZORDER`. `VACUUM` removes unreferenced files
older than the retention (7 days by default) - a storage operation, not a performance one, and it
ends time travel for that window.

```sql
OPTIMIZE main.silver.orders WHERE order_date >= date '2026-10-01' ZORDER BY (customer_id);

ALTER TABLE main.silver.orders CLUSTER BY (order_date, customer_id);
OPTIMIZE main.silver.orders;

VACUUM main.silver.orders RETAIN 168 HOURS;
```

```text
small files:  120,000 files x 3 MB   -> slow listing, more per-file stats, weak skipping
target files:     400 files x 900 MB -> one task per file, good skipping, cheap rewrites
```

Common mistakes: optimizing on a schedule instead of evidence (nightly `ZORDER` on an append-only
table), vacuuming aggressively, and partitioning by a high-cardinality column such as
`customer_id`.

## What changes between Spark 3 and Spark 4?

Most tuning advice carries over unchanged; these are the differences that show up in jobs and interviews:

+ **AQE.** Added in Spark 3.0 (coalescing, skew join, runtime join switching) and on by default since 3.2. On any modern version, start from AQE and tune `advisoryPartitionSizeInBytes` before hand-setting `spark.sql.shuffle.partitions` per query.
+ **ANSI SQL mode.** On by default in Spark 4.0 (`spark.sql.ansi.enabled=true`): invalid casts, overflow and division by zero raise errors instead of returning null. Jobs that silently produced nulls on Spark 3 can fail after an upgrade, which is a correctness fix, not a regression.
+ **Release cadence.** Spark 4.0 (May 2025) was followed by 4.1, 4.2 and 4.3; from 4.3 (October 2026) feature releases are quarterly, so read each release's behavior-change notes before upgrading a cluster.
+ **Event logs.** Spark 4 rolls event logs by default, so each application writes a directory of `events_*` files instead of one file. The `event_log_files` helper in the monitoring section handles both layouts.
+ **Runtime.** Spark 4 requires Java 17 or 21 and Scala 2.13, and drops Java 8 and 11. Executor JVM flags (`-XX:+UseG1GC` and friends) carry over; old Java 8 GC flags do not.
+ **Spark Connect.** Spark 4 ships the thin client as a first-class option. With a remote client, `df._jdf` and `sparkContext` are not available, so plan inspection goes through `explain` and the UI rather than JVM internals. PySpark 1 shows how to connect with `SparkSession.builder.remote(...)`.
+ **Python.** PySpark 4.0 needs Python 3.9 or newer, later 4.x releases raise that floor (Spark 4.3 drops Python 3.10), and pandas UDFs need `pandas` and `pyarrow` on every worker in both versions.

## Tuning exercises

Standard-library models of the quantities you read from the UI and `explain`: shuffle bytes,
broadcast thresholds, partition row counts, and executor arithmetic. The asserts check the
formula, not a real cluster.

### Try it 1. Choose shuffle partitions

**Task.** A shuffle moves 640 GB (binary units). Compute the number of shuffle partitions needed for a 128 MB target partition size, then for a 256 MB target.

**Hint.** `partitions = ceil(shuffle_bytes / target_bytes)`. One GB here is `1024` MB.

**Expected result.** `5,120` partitions for 128 MB and `2,560` for 256 MB.

In [ ]:
import math


def shuffle_partitions(shuffle_gb, target_mb):
    shuffle_mb = shuffle_gb * 1024
    return math.ceil(shuffle_mb / target_mb)


partitions_128 = shuffle_partitions(640, 128)
partitions_256 = shuffle_partitions(640, 256)
print(f"640 GB at 128 MB/partition -> {partitions_128:,} partitions")
print(f"640 GB at 256 MB/partition -> {partitions_256:,} partitions")

assert partitions_128 == 5120
assert partitions_256 == 2560
assert partitions_128 == 2 * partitions_256

### Try it 2. Broadcast threshold decision

**Task.** Using the default auto-broadcast threshold of 10 MB, decide whether a 7.5 MB dimension and a 45 MB dimension would be broadcast. Then show the driver and network cost of forcing the 45 MB dimension to broadcast across 60 executors.

**Hint.** Compare bytes against `10 * 1024 * 1024`. A forced broadcast replicates the relation to every executor.

**Expected result.** The 7.5 MB dimension broadcasts, the 45 MB one does not, and forcing it moves about `2,700` MB in total.

In [ ]:
MB = 1024 * 1024
THRESHOLD = 10 * MB
EXECUTORS = 60


def would_broadcast(size_mb):
    return size_mb * MB <= THRESHOLD


small, large = 7.5, 45.0
print(f"{small} MB -> broadcast: {would_broadcast(small)}")
print(f"{large} MB -> broadcast: {would_broadcast(large)}")

network_mb = large * EXECUTORS
print(f"forced broadcast network = {network_mb:,.0f} MB")

assert would_broadcast(small) is True
assert would_broadcast(large) is False
assert network_mb == 2700.0

### Try it 3. Detect skew from per-partition row counts

**Task.** Given the row counts of eight shuffle partitions, compute the median, the maximum, and the ratio of max to median. Flag the stage as skewed when the max exceeds five times the median.

**Hint.** Sort the counts and take the middle value, averaging the two middle values for an even count.

**Expected result.** Median `101`, maximum `5,000`, ratio about `49.5`, and the stage is skewed.

In [ ]:
counts = [100, 105, 98, 102, 100, 5000, 99, 103]


def median(values):
    ordered = sorted(values)
    mid = len(ordered) // 2
    if len(ordered) % 2 == 1:
        return ordered[mid]
    return (ordered[mid - 1] + ordered[mid]) / 2


med = median(counts)
peak = max(counts)
ratio = peak / med
skewed = ratio > 5

print(f"median = {med:.1f}, max = {peak}, ratio = {ratio:.1f}")
print("skewed:", skewed)

assert med == 101
assert peak == 5000
assert round(ratio, 1) == 49.5
assert skewed is True

### Try it 4. Salt a hot key and recompute the distribution

**Task.** One million rows are keyed by customer. The key `guest` holds 40 percent and four normal keys split the rest evenly. Compute the largest partition share before salting, then salt `guest` into 32 buckets evenly and recompute it.

**Hint.** After salting, the largest partition is either a salted bucket of the hot key or one of the normal keys, whichever is larger.

**Expected result.** The largest share falls from `0.40` to `0.15`, roughly a `2.67x` improvement.

In [ ]:
TOTAL = 1_000_000
SALT_BUCKETS = 32
hot = int(TOTAL * 0.40)
normal = (TOTAL - hot) / 4

before = hot / TOTAL
salted_bucket = hot / SALT_BUCKETS
after = max(salted_bucket, normal) / TOTAL

print(f"before: hot key share = {before:.2f}")
print(f"after : largest share = {after:.2f} (salted bucket = {salted_bucket:,.0f} rows)")
print(f"improvement = {before / after:.2f}x")

assert before == 0.40
assert after == 0.15
assert round(before / after, 2) == 2.67

### Try it 5. Decide repartition versus coalesce

**Task.** After a filter, a DataFrame has 1,000 partitions holding 4 GB in total and the downstream step wants 16 tasks. Decide between `coalesce(16)` and `repartition(16)` using a 128 MB target partition size. Repeat for a 512 MB result with 4 target tasks.

**Hint.** `coalesce` merges whole upstream partitions without a shuffle, so it is safe only when the merged partition fits the target. Otherwise you need `repartition`.

**Expected result.** 4 GB into 16 tasks is 256 MB each, so use `repartition`; 512 MB into 4 tasks is 128 MB each, so `coalesce` is fine.

In [ ]:
MB = 1024 * 1024


def choose_op(total_mb, target_tasks, target_mb=128):
    per_task = total_mb / target_tasks
    choice = "coalesce" if per_task <= target_mb else "repartition"
    return choice, per_task


choice_a, per_a = choose_op(4 * 1024, 16)
choice_b, per_b = choose_op(512, 4)
print(f"4096 MB / 16 = {per_a:.0f} MB per task -> {choice_a}")
print(f" 512 MB /  4 = {per_b:.0f} MB per task -> {choice_b}")

assert choice_a == "repartition"
assert choice_b == "coalesce"

### Try it 6. Estimate executor count from cores and memory

**Task.** A cluster has 10 nodes with 32 cores and 256 GB of RAM each. Leave one core and 1 GB per node for the operating system and the NodeManager. With 5 cores per executor, compute executors per node, total executors, total cores used, and the memory per executor.

**Hint.** `executors_per_node = floor((cores_per_node - 1) / cores_per_executor)`, then divide the usable memory by that count.

**Expected result.** `6` executors per node, `60` in total, `30` cores used per node, and about `42` GB of memory each.

In [ ]:
NODES = 10
CORES_PER_NODE = 32
MEM_PER_NODE_GB = 256
CORES_PER_EXECUTOR = 5

usable_cores = CORES_PER_NODE - 1
usable_mem = MEM_PER_NODE_GB - 1

executors_per_node = usable_cores // CORES_PER_EXECUTOR
cores_used = executors_per_node * CORES_PER_EXECUTOR
mem_per_executor = usable_mem // executors_per_node
total_executors = executors_per_node * NODES

print(f"executors per node = {executors_per_node}")
print(f"total executors    = {total_executors}")
print(f"cores used per node= {cores_used} of {CORES_PER_NODE}")
print(f"memory per executor= {mem_per_executor} GB")

assert executors_per_node == 6
assert total_executors == 60
assert cores_used == 30
assert mem_per_executor == 42

### Try it 7. Compute AQE coalescing of small partitions

**Task.** A shuffle produces 200 partitions of 2 MB each. AQE coalesces adjacent partitions toward `spark.sql.adaptive.advisoryPartitionSizeInBytes = 64m`. Compute the number of partitions after coalescing and the resulting average partition size.

**Hint.** `coalesced = ceil(total_bytes / advisory_bytes)`. Each coalesced partition packs whole input partitions, so the total is unchanged.

**Expected result.** `400` MB coalesces into `7` partitions of about `57.1` MB each.

In [ ]:
import math

INPUT_PARTITIONS = 200
PARTITION_MB = 2
ADVISORY_MB = 64

total_mb = INPUT_PARTITIONS * PARTITION_MB
coalesced = math.ceil(total_mb / ADVISORY_MB)
avg_mb = total_mb / coalesced

print(f"total = {total_mb} MB across {INPUT_PARTITIONS} partitions")
print(f"coalesced -> {coalesced} partitions, average {avg_mb:.1f} MB")

assert total_mb == 400
assert coalesced == 7
assert round(avg_mb, 1) == 57.1

### Try it 8. Compare two plans by estimated shuffle bytes

**Task.** Plan A is a sort-merge join that shuffles 1.2 TB in each direction. Plan B broadcasts a 300 MB dimension to 60 executors and shuffles the fact side not at all. Compare total bytes moved across the network and compute the reduction factor.

**Hint.** Plan A moves its shuffle write plus its shuffle read. Plan B moves the broadcast size once per executor. Convert everything to MB before comparing.

**Expected result.** Plan A moves `2.4` TiB, Plan B about `18,000` MB, a reduction of roughly `139.8x`.

In [ ]:
MB_PER_GIB = 1024

plan_a_write_mb = 1.2 * 1024 * MB_PER_GIB
plan_a_read_mb = 1.2 * 1024 * MB_PER_GIB
plan_a_total = plan_a_write_mb + plan_a_read_mb

broadcast_mb = 300
executors = 60
plan_b_total = broadcast_mb * executors

reduction = plan_a_total / plan_b_total
print(f"plan A moves {plan_a_total:,.0f} MB")
print(f"plan B moves {plan_b_total:,.0f} MB")
print(f"reduction = {reduction:.1f}x")

assert round(plan_a_total / (1024 * 1024), 1) == 2.4
assert plan_b_total == 18000
assert round(reduction, 1) == 139.8

## Real-world tuning cases

### 1. A nightly job that regressed from 20 minutes to 3 hours

One tenant's data grew; stage 7 showed one task at 90 percent of the stage time plus spill -
skew plus a fixed, oversized partition count. Fix: enable AQE coalescing and skew join with a
128 MB advisory size. Runtime returned to minutes with no memory change.

### 2. A join on a hot customer key

One customer produced 40 percent of rows and AQE skew handling was not enough, so the join was
salted into 16 buckets (the salting section above). Measure the key distribution first - salting
adds a shuffle.

### 3. Small files after streaming ingestion

A streaming sink wrote thousands of tiny files per hour, and planning and file opens dominated
scans. Fix: compact with the table format (`OPTIMIZE`, `rewrite_data_files`) and cap output with
`spark.sql.files.maxRecordsPerFile` or a repartition before writing.

### 4. A broadcast join that stopped broadcasting

A "small" product dimension grew past 10 MB, so every run shuffled both sides. Fix: a deliberate
`broadcast()` hint or a higher threshold, after checking driver and executor memory.

### 5. Streaming latency that kept climbing

Each micro-batch consumed everything available and grew larger. Fix: `maxOffsetsPerTrigger` plus
a 2-minute trigger, and a watermark on stateful operators.

### 6. Driver OOM from collecting results

`collect()` and `toPandas()` on large frames killed the driver. Fix: aggregate in the cluster and
`limit()` any sample that must cross to the driver.

In every case the UI or the plan named the bottleneck before anyone changed a setting, and the
proof was a byte count or a task duration. In an interview give the numbers and the mechanism:
"stage 7 had one task at 90 percent of the stage time and spill; AQE split the skewed partition,
spill went to zero, and runtime fell from three hours to twenty minutes" - not "I enabled AQE".

## What tuning checklist should you use in interviews?

+ Inspect the Spark UI and the physical plan first.
+ Check input size, file count, and partition pruning.
+ Filter rows and select columns early; avoid Python UDFs.
+ Broadcast small dimensions, deduplicate dimensions, handle verified skew (AQE, then salting).
+ Size shuffle partitions and enable AQE.
+ `repartition` to balance, `coalesce` to reduce output partitions.
+ Cache only reused, expensive frames; never `collect()` large data.
+ Compact small files and write sensible file sizes.
+ Validate with runtime, shuffle and spill metrics, and output correctness.

Strong answer: "I would not start with executor memory. I would read the plan and the UI, decide
whether the bottleneck is scan, shuffle, skew, join, spill, or output files, apply the smallest
targeted fix, and measure again." Naming a config is much weaker than naming the evidence that
justifies it.

## Key takeaways

- Fix the plan before you change configuration: read `explain`, the UI and the event log first.
- Measure before and after on the same data, one change at a time, and check the output is identical.
- One slow task out of 200 is skew, not a slow cluster. Try AQE skew join, then salt the key or isolate the hot value.
- Size shuffle partitions from bytes, broadcast only what is genuinely small, and write files of 128 MB-1 GB.
- Cache only a DataFrame that several actions reuse, and add executor memory only when spill or GC says so.

**Next:** [PySpark 8: real-world use cases and interviews](pyspark-08-real-world-and-interviews.ipynb)